# Checking my own subject list against what else CDA knows

*Standalone -- no prior vignette required.*

> **DRAFT STATUS:** specific counts and source names below are placeholders -- not yet run against the live API. Structure is the reviewed part; content needs verification.

I've been working with a cohort of 100 individuals I pulled from IDC. I'm not looking for new subjects, and I'm not trying to explore anything -- I already have my list. What I want to know is whether there's *more* data about these specific people sitting somewhere else, without retyping 100 IDs into a filter by hand.

In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

I have my IDs in a plain TSV, `mydatafile.tsv`, with a `subject` column. `match_from_file` takes three things: which file, which column in that file, and which CDA column to match it against:

In [ ]:
from cdapython import get_subject_data

get_subject_data(
    match_from_file={
        'input_file': 'mydatafile.tsv',
        'input_column': 'subject',
        'cda_column_to_match': 'subject_id'
    },
    add_columns='upstream_identifiers.*'
)

Most of my 100 subjects turned up in CDA at all, which is itself useful to know. Adding `upstream_identifiers.*` shows me exactly which sources have data on each one -- it turns out several of them have data spread across GC, GDC, and PDC too, not just the IDC I started with. I didn't know that until just now; I only ever submitted these people to IDC myself.

Before pulling full records, I want a shape of what I'd actually be getting:

In [ ]:
from cdapython import summarize_subjects, summarize_files

summarize_files(match_from_file={
    'input_file': 'mydatafile.tsv',
    'input_column': 'subject',
    'cda_column_to_match': 'subject_id'
})

In [ ]:
summarize_subjects(match_from_file={
    'input_file': 'mydatafile.tsv',
    'input_column': 'subject',
    'cda_column_to_match': 'subject_id'
})

Now that I know there's more data out there on these people than I submitted myself, I want to know exactly where it came from -- not just "GC has something," but a pointer back to GC's own record for this specific subject, so I can look at it directly or cite it properly:

In [ ]:
subjects_with_refs = get_subject_data(
    match_from_file={
        'input_file': 'mydatafile.tsv',
        'input_column': 'subject',
        'cda_column_to_match': 'subject_id'
    },
    include_external_refs=True
)
subjects_with_refs

There's a new `external_reference_data` column here -- a small table per subject pointing back to the actual upstream record that described them, separate from `data_source` (which just tells me *which* data center a row came from, not the specific record). This is the kind of thing I'd want before citing a source properly in a publication, or before trusting that CDA's version of a record matches what the original data center actually says.